# GMTI-Net V3 — NTIRE 2026 'Extreme PSNR' Master Notebook

This notebook provides a production-grade, end-to-end pipeline for the **GMTI-Net V3** architecture. V3 is precision-engineered for maximum PSNR using:

1. **RAFT-Style Iterative Flow Refinement**: Sub-pixel alignment for micro-motions.
2. **DCT-Domain Frequency Supervision**: Edge/Texture preservation in the frequency domain.
3. **5x5 Learned Local Kernel Synthesis**: Pixel-wise adaptive synthesis for sampling residual correction.
4. **Confidence-Weighted Self-Ensemble**: Soft-maxed fusion of multi-transform predictions.

## Quick Start
1. Ensure your GPU is connected (RTX 4060/Laptop 8GB recommended).
2. Run cells sequentially to setup, train (3 stages), and evaluate.

In [ ]:
# Cell 1 — Global Configuration
import os, sys, torch, yaml
from pathlib import Path
import numpy as np

# Determine repo ROOT
cwd = Path(os.path.abspath(""))
REPO_DIR = cwd.parent if cwd.name == "docs" else cwd
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_DIR = REPO_DIR / "checkpoints"
DATA_DIR = REPO_DIR / "data"
NTIRE_TRAIN_DIR = DATA_DIR / "NTIRE/train"
NTIRE_VAL_DIR = DATA_DIR / "NTIRE/val"

# Global Vis Frequency
VIS_FREQ = 1000

print(f"Environment Ready: {REPO_DIR}")
if torch.cuda.is_available():
    print(f"Device: {DEVICE} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")
else:
    print("Device: CPU")

## 1. Setup & Dependencies
Installs optimized libraries for PSNR evaluation and visualization.

In [ ]:
# Cell 2 — Automated dependency install
import subprocess
packages = ["lpips", "pytorch-msssim", "scikit-image", "psutil", "tqdm", "pyyaml", "opencv-python", "matplotlib"]
subprocess.run([sys.executable, "-m", "pip", "install"] + packages, check=False)
print("Dependencies synchronized.")

## 2. Dataset Alignment
Detects NTIRE/Vimeo90K and handles validation split preparation.

In [ ]:
# Cell 3 — Dataset Verification
import shutil

def ensure_val_split():
    if not NTIRE_VAL_DIR.exists() or not any(NTIRE_VAL_DIR.glob("vid_*")):
        print("Auto-creating NTIRE validation split...")
        NTIRE_VAL_DIR.mkdir(parents=True, exist_ok=True)
        all_train_vids = sorted(list(NTIRE_TRAIN_DIR.glob("vid_*")))
        if len(all_train_vids) > 5:
            # Reserve last 5 videos for internal validation
            for v in all_train_vids[-5:]:
                shutil.move(str(v), str(NTIRE_VAL_DIR / v.name))
            print(f"Moved {len(all_train_vids[-5:])} videos to {NTIRE_VAL_DIR}")

if NTIRE_TRAIN_DIR.exists():
    ensure_val_split()
    print("NTIRE Dataset: Detected & Prepared.")
else:
    print("WARNING: NTIRE data missing. Synthetic mode will be used if triggered.")

## 3. V3 Architecture & High-Fidelity Sanity
Verifies the RAFT-style flow and Kernel Synthesis heads are operational with V3 visual analytics.

In [ ]:
# Cell 4 — Model Smoke Test with Analytics Grid
from models.gmti_net import GMTINet
from utils.vis_v3 import create_analytics_grid, visualize_kernels, visualize_dct_spectrum
import yaml

with open("config.yaml", "r") as f: config = yaml.safe_load(f)
m_cfg = config["model"]

model = GMTINet(
    swin_depth=m_cfg["encoder"]["swin_depth"],
    swin_heads=m_cfg["encoder"]["swin_heads"],
    flow_refinement_iters=m_cfg["flow"]["refinement_iters"],
    num_hypotheses=m_cfg["flow"]["num_hypotheses"],
    use_deformable=m_cfg["flow"]["use_deformable"],
    transformer_blocks=m_cfg["transformer"]["blocks"],
    transformer_heads=m_cfg["transformer"]["heads"],
    transformer_dim=m_cfg["transformer"]["embed_dim"]
).to(DEVICE).eval()

with torch.no_grad():
    L, R = torch.randn(1, 3, 256, 256).to(DEVICE), torch.randn(1, 3, 256, 256).to(DEVICE)
    M = torch.randn(1, 3, 256, 256).to(DEVICE)
    pred, aux = model(L, R)

    print(f"Model Output Shape: {pred.shape}")
    print(f"Sanity Check (Analytics Grid)... ")
    os.makedirs("visualizations", exist_ok=True)
    create_analytics_grid(L, R, pred, M, aux, 0)

    # Show DCT Spectrum
    spec_fig = visualize_dct_spectrum(pred)
    spec_fig.show()

    print("V3.1 Pro Sanity Check: PASSED (Analytics row 2 now shows Accel Map)")

## 4. Staged Curriculum Training
Each cell runs one of the three critical NTIRE stages. 
**Note:** Training now includes the **Enhanced Progress Bar** showing Stage, PSNR, and VRAM usage.

In [ ]:
# Cell 5 — Stage 1: Motion Pre-training (256x256)
!python train.py --config config.yaml --stage 1 --resume checkpoints/latest.pth

In [ ]:
# Cell 6 — Stage 2: Structural Refinement (384x384)
!python train.py --config config.yaml --stage 2 --resume checkpoints/best_ema.pth

In [ ]:
# Cell 7 — Stage 3: Extreme PSNR Finetune (512x512)
!python train.py --config config.yaml --stage 3 --resume checkpoints/best_ema.pth

## 5. Advanced Evaluation & Submission Prep
Uses **Snapshot Averaging** and **Confidence-Weighted Self-Ensemble**.

In [ ]:
# Cell 8 — Snapshot Averaging & Ensemble Benchmarking
import glob
ckpts = sorted(list(CHECKPOINT_DIR.glob("iter_*.pth")))[-15:] # Last 15 checkpoints
ckpt_list = " ".join([str(c) for c in ckpts])

print("Running Confidence-Weighted Ensemble Benchmarking...")
try:
    !python scripts/benchmark.py --input_dir {NTIRE_VAL_DIR} --output_dir results --snapshots {ckpt_list} --self_ensemble --scales 1.0 1.25
except Exception as e:
    print("Benchmarking failed, ensure stage-3 is complete.")

## 6. Extreme Analytics Dashboard
Deep-dive into the model's internals. Inspect learned kernels and frequency preservation.

In [ ]:
# Cell 9 — Interactive V3.1 Pro Analytics
import matplotlib.pyplot as plt
from PIL import Image
import glob

def show_latest_analytics():
    files = sorted(glob.glob("visualizations/analytics_iter_*.png"))
    if not files:
        print("No analytics images found. Run at least 1000 iterations.")
        return

    latest = files[-1]
    img = Image.open(latest)
    plt.figure(figsize=(25, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"V3.1 Pro Analytics Dash: {os.path.basename(latest)}")
    plt.show()

    print("Row 1: Input L | Pred | GT | Input R | Boosted Error")
    print("Row 2: Flow L->M | Accel Map | Conf L | Conf R | Fuse/Occ Mask")

show_latest_analytics()

## 7. Submission Synthesis
Compresses the results into the required NTIRE 2026 format.

In [ ]:
# Cell 10 — Final ZIP Generation
!python scripts/prepare_submission.py --results_dir results --output Submission_GMTI_V3_Pro.zip
print('Submission ready: Submission_GMTI_V3_Pro.zip')